# Notebook 12 — Golden Baseline Strategy (Briefing Alignment)

Two-step approach to satisfy **<5% train–test gap** while targeting **F1 weighted ≥ 0.80**:

| Step | Model | Purpose |
|------|--------|--------|
| **1** | Toxic-BERT (all layers **frozen**) | Esencial baseline — no fine-tuning on 1k samples; gap ≈ 0% |
| **2** | Last **2** layers + **R-Drop**, lr **5e-6**, 15 epochs | Performance squeeze — F1 toward 0.80, gap ≤ 4.9% |
| **3** | Hybrid + LR (**C=0.001**, **200** features) | Safety net — stable LR pulls hybrid gap under 5% |

```bash
uv run python -m src.pipeline.run_golden_baseline_pipeline
```

## 0. Setup

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

cfg_path = PROJECT_ROOT / "configs" / "golden_baseline_training.yaml"
cfg = yaml.safe_load(open(cfg_path))
reports_dir = PROJECT_ROOT / "reports" / "golden_baseline"
print(f"Config: {cfg_path.name}")
print(f"Augmentation: {cfg['augmentation']['enabled']}")
print(f"Squeeze: last {cfg['transformer']['train_last_n_layers']} layers, R-Drop={cfg['transformer']['rdrop']['enabled']}")

Config: golden_baseline_training.yaml
Augmentation: False
Squeeze: last 2 layers, R-Drop=True


## 1. Run pipeline (Steps 1–3)

In [2]:
from src.pipeline.run_golden_baseline_pipeline import run_golden_baseline_pipeline

metrics = run_golden_baseline_pipeline(config_path=cfg_path)
run_id = metrics["run_id"]

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-24 21:33:42 | INFO     | src.pipeline.run_golden_baseline_pipeline | ============================================================
2026-05-24 21:33:42 | INFO     | src.pipeline.run_golden_baseline_pipeline | GOLDEN BASELINE STRATEGY — run=20260524_213342
2026-05-24 21:33:42 | INFO     | src.pipeline.run_golden_baseline_pipeline | ============================================================
2026-05-24 21:33:42 | INFO     | src.data.loader | Cargando dataset: /Users/miraekang/proyectos/ai-nlp/data/raw/youtoxic_english_1000.csv
2026-05-24 21:33:42 | INFO     | src.data.loader |   Shape: (1000, 15)
2026-05-24 21:33:42 | INFO     | src.data.loader |   Columnas validadas ✅
2026-05-24 21:33:42 | WARNING  | src.data.loader |   3 duplicados eliminados
2026-05-24 21:33:42 | INFO     | src.data.loader |   Toxicos: 459 (46.0%)
2026-05-24 21:33:42 | INFO     | src.data.dual_loader | Loading preprocessed text: /Users/miraekang/proyectos/ai-nlp/data/processed/v2/comments_preprocessed.csv
2026-

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10445.87it/s]

2026-05-24 21:33:43 | INFO     | src.models.transformer_trainer | Inference-only — all 12 encoder blocks + head frozen (zero fine-tuning)


2026-05-24 21:33:44 | INFO     | src.models.transformer_trainer | Golden Baseline — unitary/toxic-bert (inference only, no training)


...

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 677/677 [00:00<00:00, 25954.90 examples/s]
/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


2026-05-24 21:35:46 | INFO     | src.pipeline.run_golden_baseline_pipeline | Step 3 — Hybrid Safety Net (LR C=0.001, max_features=200)
2026-05-24 21:35:46 | INFO     | src.models.metadata_lr | Metadata LR trained — C=0.001 | tfidf_dim=200 | meta_dim=5
2026-05-24 21:35:46 | INFO     | src.models.metadata_lr | Metadata LR saved: /Users/miraekang/proyectos/ai-nlp/models/golden_squeeze_lr.joblib
2026-05-24 21:35:46 | INFO     | src.pipeline.run_golden_baseline_pipeline | Report: /Users/miraekang/proyectos/ai-nlp/reports/golden_baseline/integrated_report_20260524_213342.md
2026-05-24 21:35:46 | INFO     | src.pipeline.run_golden_baseline_pipeline | ============================================================
2026-05-24 21:35:46 | INFO     | src.pipeline.run_golden_baseline_pipeline | BASELINE  F1w=0.7903 gap_pp=0.16 (✅ <1%)
2026-05-24 21:35:46 | INFO     | src.pipeline.run_golden_baseline_pipeline | HYBRID    F1w=0.7479 gap_pp=4.39 (⚠️ below target)
2026-05-24 21:35:46 | INFO     | src.pipe

## 2. Briefing compliance summary

In [3]:
def _row(key, label):
    m = metrics.get(key, {})
    if not m:
        return None
    return {
        "step": label,
        "f1_test": m.get("f1_weighted"),
        "gap_pp": m.get("train_test_gap_pp"),
        "gap_ok": m.get("gap_ok", m.get("esencial_gap_ok", False)),
        "f1_target_ok": (m.get("f1_weighted") or 0) >= metrics.get("target_f1_weighted", 0.8),
    }

rows = [
    _row("golden_baseline", "1 — Golden Baseline"),
    _row("performance_squeeze", "2 — Performance Squeeze"),
    _row("hybrid_safety_net", "3 — Hybrid Safety Net"),
]
summary = pd.DataFrame([r for r in rows if r])
summary

,step,f1_test,gap_pp,gap_ok,f1_target_ok
0,1 — Golden Baseline,0.7903,0.16,True,False
1,2 — Performance Squeeze,0.7588,2.83,False,False
2,3 — Hybrid Safety Net,0.7479,4.39,True,False


## 3. Integrated report

In [4]:
from IPython.display import Markdown, display

md_path = reports_dir / f"integrated_report_{run_id}.md"
if md_path.exists():
    display(Markdown(md_path.read_text()))
else:
    latest = sorted(reports_dir.glob("integrated_report_*.md"))[-1]
    display(Markdown(latest.read_text()))

# Golden Baseline Strategy — 20260524_213342

Two-step briefing alignment: **Esencial** frozen expert baseline, then **Experto** squeeze + hybrid.

## Step 1 — Golden Baseline (Esencial)

| Metric | Value | Target |
|--------|-------|--------|
| F1 weighted (test) | **0.7903** | ~0.72 (pretrained expert) |
| Train–test gap (pp) | **0.16** | < 1.0% ✅ |
| Fine-tuning | None (all layers frozen) | — |
| Threshold | 0.12 | val-tuned |

## Step 2 — Performance Squeeze (Experto)

| Metric | Value | Target |
|--------|-------|--------|
| F1 weighted (test) | **0.7588** | ≥ 0.8 |
| Train–test gap (pp) | **2.83** | ≤ 4.9% |
| R-Drop | True | enabled |
| Layers trained | last partial_last_2 | 2 + head |

## Step 3 — Hybrid Safety Net (Final)

| Metric | Value | Target |
|--------|-------|--------|
| F1 weighted (test) | **0.7479** | ≥ 0.8 ⚠️ |
| Train–test gap (pp) | **4.39** | < 5.0% ✅ |
| Weights | BERT 0.9 / LR 0.1 | anchor |
| LR regularization | C=0.001, max_features=200 | stability |

### Overall: ⚠️ Review gaps / F1

- JSON: `reports/golden_baseline/golden_baseline_run_20260524_213342.json`


## Conclusion

**Step 1 (Golden Baseline)** loads pretrained `unitary/toxic-bert` (6-label head, sigmoid `toxic` score) with **no fine-tuning**. Train–test gap stays **under 1%** (Esencial compliant). Holdout weighted F1 is often **~0.79**, above the ~0.72 briefing estimate, because the Jigsaw-trained head is already a strong expert.

**Step 2 (Performance Squeeze)** unfreezes the last two layers with **R-Drop** and **lr=5e-6**. Gap remains under **5%**, but F1 on 1k rows may fall below the frozen baseline if fine-tuning overfits.

**Step 3 (Hybrid Safety Net)** adds LR (**C=0.001**, **200** features) for gap stability; final hybrid F1 may trail BERT-only unless ensemble weights favor the frozen expert.

Artifacts: `models/golden_squeeze_toxic_bert/`, `models/golden_squeeze_lr.joblib`, `reports/golden_baseline/`.